In [2]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -q 2>/dev/null
!pip install selenium webdriver-manager lxml pandas pyarrow chess -q

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 1 not upgraded.
Need to get 11.2 MB/141 MB of archives.
After this operation, 476 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-data all 2.36.0-3build1 [2,824 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatk1.0-0 amd64 2.36.0-3build1 [51.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libatspi2.0-0 a

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def init_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)

driver = init_driver()
print("[OK] Chrome working")

[OK] Chrome working


In [4]:
import re, time, logging
from lxml import html
from collections import deque
from pathlib import Path
import pandas as pd

logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

def clean(text):
    text = re.sub(r"[♔♕♖♗♘♙♚♛♜♝♞♟]", "", text)
    return re.sub(r"\s+", " ", text).strip()

def get_page_source(driver, url, retries=3):
    for attempt in range(retries):
        try:
            driver.get(url)
            time.sleep(0.5)
            src = driver.page_source
            if "cf-subheadline" in src or "enable_cookies" in src:
                time.sleep(3); continue
            return src
        except Exception as e:
            print(f"  [Error #{attempt+1}] {e}"); time.sleep(3)
    return None

def parse_page(current_url, source):
    tree = html.fromstring(source)

    wanted = {
        "Best Moves": "best_moves",
        "Important Alternatives": "important_alternatives",
        "Critical Mistakes": "critical_mistakes",
    }

    result = {
        "best_moves": "",
        "important_alternatives": "",
        "critical_mistakes": "",
    }

    def clean_html_text(fragment_html):
        fragment_html = re.sub(r'<svg[^>]*>.*?</svg>', ' ', fragment_html, flags=re.DOTALL)
        fragment_html = re.sub(r'<a\s[^>]*>(.*?)</a>', r'\1', fragment_html, flags=re.DOTALL)
        text = re.sub(r'<[^>]+>', ' ', fragment_html)
        return clean(text)

    for h2 in tree.xpath('//h2'):
        title = clean(" ".join(h2.xpath('.//text()')))
        if title not in wanted:
            continue

        parts = []
        for sib in h2.itersiblings():
            if getattr(sib, "tag", None) == "h2":
                break
            raw = html.tostring(sib, encoding="unicode")
            txt = clean_html_text(raw)
            if txt:
                parts.append(txt)

        result[wanted[title]] = "\n".join(parts)

    return result

    for h2 in tree.xpath('//h2'):
        title = clean(" ".join(h2.xpath('.//text()')))
        if title not in wanted:
            continue

        parts = []
        for sib in h2.itersiblings():
            if getattr(sib, "tag", None) == "h2":
                break
            raw = html.tostring(sib, encoding="unicode")
            txt = clean_html_text(raw)
            if txt:
                parts.append(txt)

        result[wanted[title]] = "\n".join(parts)

    base_path = current_url.replace("https://www.schachzeit.com", "").rstrip("/")
    seen, children = set(), []
    for href in tree.xpath('//a[contains(@href,"/with/")]/@href'):
        suffix = href[len(base_path):]
        if not re.match(r'^/with/[^/]+$', suffix):
            continue
        full_url = "https://www.schachzeit.com" + href
        if full_url not in seen:
            seen.add(full_url)
            children.append((full_url, href.rsplit("/with/", 1)[-1]))

    result["children"] = children
    return result

def format_text(parsed):
    parts = []

    if parsed["best_moves"]:
        parts.append("[Best Moves]\n" + parsed["best_moves"])

    if parsed["important_alternatives"]:
        parts.append("[Important Alternatives]\n" + parsed["important_alternatives"])

    if parsed["critical_mistakes"]:
        parts.append("[Critical Mistakes]\n" + parsed["critical_mistakes"])

    return "\n\n".join(parts)

def get_seed_urls(driver):
    src = get_page_source(driver, "https://www.schachzeit.com/en/openings")
    if not src: return []
    tree = html.fromstring(src)
    seen, seeds = set(), []
    for h in tree.xpath('//a/@href'):
        if h.startswith('/en/openings/'):
            full = "https://www.schachzeit.com" + h
        elif h.startswith('openings/'):
            full = "https://www.schachzeit.com/en/" + h
        else:
            continue
        if '/with/' in full: continue
        if full not in seen:
            seen.add(full); seeds.append(full)
    print(f"[Seeds] {len(seeds)} дебютов")
    return seeds

def _save(results, output_path, visited, visited_file):
    if not results: return
    df = pd.DataFrame(results)
    df.to_csv(f"{output_path}_checkpoint.tsv", sep='\t', index=False)
    df.to_csv(f"{output_path}.tsv", sep='\t', index=False)
    df.to_parquet(f"{output_path}.parquet", index=False)
    visited_file.write_text("\n".join(visited))

def crawl_openings_only(init_driver, output_path="openings_text", delay=0.5, checkpoint_every=200):
    ckpt_file = Path(f"{output_path}_checkpoint.tsv")
    visited_file = Path(f"{output_path}_visited.txt")

    results, visited = [], set()

    if ckpt_file.exists():
        done_df = pd.read_csv(ckpt_file, sep="\t")
        results = done_df.to_dict("records")
        visited = {r["url"] for r in results}
        print(f"[Resume] {len(results)} pages checkpoint")

    if visited_file.exists():
        visited |= set(visited_file.read_text().splitlines())

    driver = init_driver()
    seeds = get_seed_urls(driver)
    seeds = [url for url in seeds if url not in visited]

    processed = len(results)
    print(f"[Openings] to go: {len(seeds)} | Now: {processed}")

    try:
        for url in seeds:
            try:
                source = get_page_source(driver, url)
            except Exception as e:
                print(f"[Driver error] {e}")
                try:
                    driver.quit()
                except:
                    pass
                driver = init_driver()
                source = get_page_source(driver, url)

            if not source:
                print(f"[Skip] {url}")
                continue

            parsed = parse_page(url, source)
            text = format_text(parsed)

            visited.add(url)
            results.append({
                "url": url,
                "opening_key": url.split("/openings/")[-1],
                "best_moves": parsed["best_moves"],
                "important_alternatives": parsed["important_alternatives"],
                "critical_mistakes": parsed["critical_mistakes"],
                "text": text,
            })

            processed += 1
            bm = parsed["best_moves"].replace("\n", " ")[:80]
            ia = parsed["important_alternatives"].replace("\n", " ")[:80]
            cm = parsed["critical_mistakes"].replace("\n", " ")[:80]
            status = "✓" if text else "✗"

            print(
                f"[{processed}] {status} {url.split('/openings/')[-1][:60]} | "
                f"BM: {bm} | IA: {ia} | CM: {cm}"
            )

            if processed % checkpoint_every == 0:
                _save(results, output_path, visited, visited_file)
                print(f"  → Checkpoint {processed}")

            time.sleep(delay)

        _save(results, output_path, visited, visited_file)
        return pd.DataFrame(results)

    finally:
        try:
            driver.quit()
        except:
            pass

In [5]:
driver = init_driver()
print("[OK] Chrome working")

[OK] Chrome working


In [ ]:
result_df = crawl_openings_only(init_driver, delay=0.5, checkpoint_every=200)

[Seeds] 2896 дебютов
[Openings] К обработке: 2896 | Уже: 0
[1] ✗ starting-position | BM:  | IA:  | CM: 
[2] ✓ amar-opening | BM: The Humble d5 d5 is Black’s strongest response, establishing control over the ce | IA: The Flexible Nc6 Nc6 develops a piece to a natural post while supporting potenti | CM: The Defensive f6 f6 weakens Black's kingside pawn structure and unnecessarily cr
[3] ✓ amar-opening/paris-gambit | BM: The Strategic e4 The move e4 is Black’s most principled response, taking control | IA: The Curious exf4 This move accepts the gambit and exposes the e1-a5 diagonal , w | CM: The Misguided Nc6 The move Nc6 aims to develop a knight but fails to press any i
[4] ✓ amar-opening/paris-gambit/gent-gambit | BM: The Reliable Nf6 The move Nf6 is crucial in developing Black’s position harmonio | IA: The Guarding h6 Though not the most optimal, h6 strategically makes sense by pre | CM: The Misguided Nd7 Nd7 is a strategic error as it blocks the path for the bishop 
[5] ✓ amsterdam-at

In [ ]:
from google.colab import files
files.download("openings_text.parquet")
files.download("openings_text.tsv")
files.download("openings_text_checkpoint.tsv")
files.download("openings_text_visited.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>